In [2]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
import boto3
from botocore.client import Config
from datetime import datetime
import io

DB_CONFIG = {
    'host': 'postgres',
    'port': 5432,
    'database': 'oil_gas_db',
    'user': 'analyst',
    'password': 'analyst123'
}

def get_db_engine():
    connection_string = f"postgresql://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
    return create_engine(connection_string)

MINIO_CONFIG = {
    'endpoint': 'minio:9000',
    'access_key': 'minioadmin',
    'secret_key': 'minioadmin123',
    'bucket': 'oil-gas-data',
    'secure': False
}

def get_minio_client():
    return boto3.client(
        's3',
        endpoint_url=f"http://{MINIO_CONFIG['endpoint']}",
        aws_access_key_id=MINIO_CONFIG['access_key'],
        aws_secret_access_key=MINIO_CONFIG['secret_key'],
        config=Config(signature_version='s3v4'),
        region_name='us-east-1'
    )

def ensure_bucket():
    s3 = get_minio_client()
    try:
        s3.head_bucket(Bucket=MINIO_CONFIG['bucket'])
    except:
        s3.create_bucket(Bucket=MINIO_CONFIG['bucket'])
        print(f"Bucket {MINIO_CONFIG['bucket']} created")

engine = get_db_engine()

tables = ['wells', 'production', 'well_telemetry', 'well_targets', 
          'pumps', 'pump_sensors', 'pump_failures', 'deliveries', 
          'drivers', 'vehicles', 'oil_stations']

dataframes = {}
for table in tables:
    print(f"Loading {table}...")
    dataframes[table] = pd.read_sql(f"SELECT * FROM {table}", engine)
    print(f"  {len(dataframes[table])} rows loaded")

s3 = get_minio_client()
ensure_bucket()

prod_df = dataframes['production']
prod_df['date'] = pd.to_datetime(prod_df['date'])

for date, group in prod_df.groupby(prod_df['date'].dt.date):
    partition_key = date.strftime('year=%Y/month=%m/day=%d')
    parquet_buffer = io.BytesIO()
    group.to_parquet(parquet_buffer, index=False)
    s3.put_object(
        Bucket=MINIO_CONFIG['bucket'],
        Key=f"production/{partition_key}/data.parquet",
        Body=parquet_buffer.getvalue()
    )
    print(f"Saved production for {date}")

telemetry_df = dataframes['well_telemetry']
telemetry_df['timestamp'] = pd.to_datetime(telemetry_df['timestamp'])
telemetry_df['date'] = telemetry_df['timestamp'].dt.date

for date, group in telemetry_df.groupby('date'):
    partition_key = pd.Timestamp(date).strftime('year=%Y/month=%m/day=%d')
    parquet_buffer = io.BytesIO()
    group.drop('date', axis=1).to_parquet(parquet_buffer, index=False)
    s3.put_object(
        Bucket=MINIO_CONFIG['bucket'],
        Key=f"well_telemetry/{partition_key}/data.parquet",
        Body=parquet_buffer.getvalue()
    )
    print(f"Saved telemetry for {date}")

for table in ['wells', 'well_targets', 'pumps', 'drivers', 'vehicles']:
    csv_buffer = io.StringIO()
    dataframes[table].to_csv(csv_buffer, index=False)
    s3.put_object(
        Bucket=MINIO_CONFIG['bucket'],
        Key=f"reference/{table}.csv",
        Body=csv_buffer.getvalue().encode('utf-8')
    )
    print(f"Saved {table}")

print("ETL completed successfully!")

Loading wells...
  5 rows loaded
Loading production...
  150 rows loaded
Loading well_telemetry...
  48 rows loaded
Loading well_targets...
  90 rows loaded
Loading pumps...
  5 rows loaded
Loading pump_sensors...
  72 rows loaded
Loading pump_failures...
  3 rows loaded
Loading deliveries...
  30 rows loaded
Loading drivers...
  5 rows loaded
Loading vehicles...
  5 rows loaded
Loading oil_stations...
  20 rows loaded
Saved production for 2025-10-01
Saved production for 2025-10-02
Saved production for 2025-10-03
Saved production for 2025-10-04
Saved production for 2025-10-05
Saved production for 2025-10-06
Saved production for 2025-10-07
Saved production for 2025-10-08
Saved production for 2025-10-09
Saved production for 2025-10-10
Saved production for 2025-10-11
Saved production for 2025-10-12
Saved production for 2025-10-13
Saved production for 2025-10-14
Saved production for 2025-10-15
Saved production for 2025-10-16
Saved production for 2025-10-17
Saved production for 2025-10-18
S